# Biohub Gold v1 — FIRST diagnostic submit (H-002 image policy + gate-10 links)

Config provenance (repo `davidlueng986-alt/kaggle-biohub-cell-tracking-rd`, PROTOCOL v1.1):
- Detection: DoG σ=(1,3,3)/(1.6,5,5), per-embryo percentile (44b6→99.0, 6bba→98.5, unknown→98.5 fallback). Test embryos are disjoint from train, so the fallback governs. Standing image policy worst-adj ~0.82 (EXP-0010).
- Linking: causal adjacent-frame Hungarian, gate 10 µm (EXP-0018 PROMOTED on oracle arm). Caveat (EXP-0017 arm2): gate-10 was −0.006 vs gate-7 on one dense image sample — LB diagnostic arbitrates.
- NO fork proposals (safety): r10 combo unpromoted (EXP-0020 denial); division term expected 0. Division-trap avoidance per PROTOCOL pitfalls.
- Provenance guards baked in: per-video global re-id (EXP-0016 lesson), consecutive CSV ids, edge-ref validation, per-video timing with 12 h projection.
- Offline, CPU numpy/scipy/zarr only. Public LB is DIAGNOSTIC (PROTOCOL §4); promotion never on LB alone.

In [ ]:
import importlib, os, subprocess, sys
WHEELS = "/kaggle/input/biohub-zarr-wheels"
try:
    import zarr; print("zarr present", zarr.__version__)
except ImportError:
    assert os.path.isdir(WHEELS), f"FATAL: no zarr and no wheels dir {WHEELS}"
    env = dict(os.environ, PIP_NO_INDEX="1", PIP_DISABLE_PIP_VERSION_CHECK="1", PIP_NO_BUILD_ISOLATION="1")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "--no-index", "--find-links", WHEELS, "zarr", "numcodecs"], env=env)
    import zarr; print("zarr installed offline", zarr.__version__)
import numcodecs; print("numcodecs", numcodecs.__version__)

In [ ]:
import os, sys
for mod in ("numpy", "scipy", "zarr"):
    try: __import__(mod); print(mod, "OK", __import__(mod).__version__ if hasattr(__import__(mod), "__version__") else "")
    except ImportError: raise SystemExit(f"FATAL: {mod} not preinstalled in Kaggle image — aborting")
import numpy as np
def find_test_root(base="/kaggle/input", depth=4):
    """Dir whose children are *.zarr samples (robust to mount layout)."""
    hits = {}
    for root, dirs, _ in os.walk(base):
        if root.count(os.sep) - base.count(os.sep) > depth:
            dirs[:] = []
            continue
        zs = sorted(dd for dd in dirs if dd.endswith(".zarr"))
        if zs:
            hits[root] = zs
    print("input top tree:", sorted(os.listdir(base)))
    assert hits, "FATAL: no .zarr found under /kaggle/input"
    cands = [r for r in hits if os.path.basename(r) == "test" and hits[r]]
    best = max(cands, key=lambda r: len(hits[r])) if cands else max(hits, key=lambda r: len(hits[r]))
    print(f"test root: {best} ({len(hits[best])} samples)")
    return best, hits[best]
TEST_ROOT, _SAMPLE_DIRS = find_test_root()
SAMPLES = sorted(dd[:-len(".zarr")] for dd in _SAMPLE_DIRS)
print(f"videos: {len(SAMPLES)}", SAMPLES[:4], "..." if len(SAMPLES) > 4 else "")
assert SAMPLES, "FATAL: empty sample list"
OUT_CSV = "/kaggle/working/submission.csv"
PCT_MAP = {"44b6": 99.0, "6bba": 98.5}
PCT_DEFAULT = 98.5
GATE_UM = 10.0
VOXEL = (1.625, 0.40625, 0.40625)
SIG_SMALL, SIG_LARGE, MIN_SIZE = (1.0, 3.0, 3.0), (1.6, 5.0, 5.0), 50


In [ ]:
from scipy.ndimage import gaussian_filter, label
try:
    from scipy.optimize import linear_sum_assignment as _lsa
    _HAVE_LSA = True
except ImportError:
    _HAVE_LSA = False
print("scipy LSA:", _HAVE_LSA)

def assign(cost):
    n = len(cost)
    if n == 0: return []
    if _HAVE_LSA and n > 60:
        ri, ci = _lsa(np.asarray(cost, dtype=float))
        out = [-1]*n
        for r, c in zip(ri.tolist(), ci.tolist()): out[r] = c
        return out
    u, v, p, way = [0.]*(n+1), [0.]*(n+1), [0]*(n+1), [0]*(n+1)
    for i in range(1, n+1):
        p[0] = i; j0 = 0
        minv, used = [float('inf')]*(n+1), [False]*(n+1)
        while True:
            used[j0] = True; i0 = p[j0]; delta, j1 = float('inf'), 0
            for j in range(1, n+1):
                if used[j]: continue
                cur = cost[i0-1][j-1]-u[i0]-v[j]
                if cur < minv[j]: minv[j], way[j] = cur, j0
                if minv[j] < delta: delta, j1 = minv[j], j
            for j in range(n+1):
                if used[j]: u[p[j]] += delta; v[j] -= delta
                else: minv[j] -= delta
            j0 = j1
            if p[j0] == 0: break
        while j0: j1 = way[j0]; p[j0] = p[j1]; j0 = j1
    a = [-1]*n
    for j in range(1, n+1):
        if p[j]: a[p[j]-1] = j-1
    return a

def um_dist(a, b):
    dz, dy, dx = (a[0]-b[0])*VOXEL[0], (a[1]-b[1])*VOXEL[1], (a[2]-b[2])*VOXEL[2]
    return (dz*dz+dy*dy+dx*dx) ** 0.5

def detect_frame(vol, pct):
    v = np.asarray(vol, dtype=np.float32)
    dog = gaussian_filter(v, SIG_SMALL) - gaussian_filter(v, SIG_LARGE)
    lab, n = label(dog >= float(np.percentile(dog, pct)))
    sizes = np.bincount(lab.ravel())
    out = []
    for i in range(1, n+1):
        if int(sizes[i]) < MIN_SIZE: continue
        z, y, x = np.argwhere(lab == i).mean(axis=0)
        out.append((int(round(z)), int(round(y)), int(round(x))))
    return sorted(out)

def link_frames(frames):
    nodes, edges, gid = [], [], [0]
    def new(t, z, y, x):
        gid[0] += 1
        nodes.append({"id": gid[0], "t": t, "z": z, "y": y, "x": x})
        return gid[0]
    prev_ids, prev_pos = [], {}
    for t, cents in enumerate(frames):
        cur_ids = [new(t, *c) for c in cents]
        cur_pos = dict(zip(cur_ids, cents))
        if prev_ids:
            P, Q = sorted(prev_ids), sorted(cur_ids)
            n, m, N = len(P), len(Q), max(len(P), len(Q))
            C = [[0.]*N for _ in range(N)]
            for i in range(N):
                for j in range(N):
                    if i < n and j < m:
                        dd = um_dist(prev_pos[P[i]], cur_pos[Q[j]])
                        C[i][j] = dd if dd <= GATE_UM else 1e9
                    elif i < n: C[i][j] = GATE_UM + 1e-9
            for i, j in enumerate(assign(C)):
                if i < n and 0 <= j < m and C[i][j] <= GATE_UM:
                    edges.append([P[i], Q[j]])
        prev_ids, prev_pos = cur_ids, cur_pos
    return nodes, sorted(edges)

In [ ]:
import csv, time, zarr
T0 = time.time(); rows, rid, times = [], [0], []
for k, name in enumerate(SAMPLES):
    t0 = time.time()
    pct = PCT_MAP.get(name.split("_")[0], PCT_DEFAULT)
    vol = zarr.open_group(os.path.join(TEST_ROOT, name + ".zarr"), mode="r")["0"]
    T = vol.shape[0]
    frames = []
    for t in range(T):
        frames.append(detect_frame(vol[t], pct))
    nodes, edges = link_frames(frames)
    for n in sorted(nodes, key=lambda d: d["id"]):
        rows.append([rid[0], name, "node", n["id"], n["t"], n["z"], n["y"], n["x"], -1, -1]); rid[0] += 1
    for u, v in sorted(map(tuple, edges)):
        rows.append([rid[0], name, "edge", -1, -1, -1, -1, -1, u, v]); rid[0] += 1
    dt = time.time() - t0; times.append(dt)
    el = time.time() - T0
    print(f"[{k+1}/{len(SAMPLES)}] {name}: pct={pct} T={T} det={len(nodes)} edges={len(edges)} {dt:.0f}s (elapsed {el/3600:.2f}h, proj {el*(len(SAMPLES)/(k+1))/3600:.1f}h)", flush=True)
with open(OUT_CSV, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id","dataset","row_type","node_id","t","z","y","x","source_id","target_id"])
    w.writerows(rows)
print(f"WROTE {OUT_CSV}: {len(rows)} rows, {len(SAMPLES)} datasets, total {(time.time()-T0)/3600:.2f}h")

In [ ]:
import csv
with open(OUT_CSV, newline="") as f:
    r = list(csv.DictReader(f))
assert [int(x["id"]) for x in r] == list(range(len(r))), "ids not consecutive"
dss, node_ids, bad = set(), {}, 0
for x in r:
    dss.add(x["dataset"])
    if x["row_type"] == "node":
        assert (x["source_id"], x["target_id"]) == ("-1", "-1"), x
        node_ids.setdefault(x["dataset"], set()).add(int(x["node_id"]))
    elif x["row_type"] == "edge":
        assert (x["node_id"], x["t"], x["z"], x["y"], x["x"]) == ("-1",)*5, x
    else: raise SystemExit(f"bad row_type {x}")
assert dss == set(SAMPLES), f"dataset mismatch: missing {set(SAMPLES)-dss}"
for x in r:
    if x["row_type"] == "edge":
        k = node_ids[x["dataset"]]
        assert int(x["source_id"]) in k and int(x["target_id"]) in k, f"dangling {x}"
print(f"SUBMISSION VALID: {len(r)} rows, {len(dss)} datasets, all refs resolve")